# TruthLens audio detector on a Colab GPU

Downloads ASVspoof 2019 LA, trains LCNN with LFCC and the self-supervised models, then evaluates them with min t-DCF and exports LCNN to ONNX.

Runs on Colab or Kaggle. On free Colab the session ends after about 90 idle minutes, so every training command stops itself after `BUDGET` minutes (75) with its checkpoint on Drive: run the cell again in a new session and it resumes. On Kaggle, turn on Internet and a GPU in the settings panel, then use Save Version, Save & Run All: it runs unattended for up to 12 hours with no idle limit, and the results land in the notebook's Output tab.

## Setup

The notebook clones the repository from GitHub, so push your local commits first. Only cells marked GPU only need a GPU runtime. Everything that must survive a disconnect is written to Google Drive.

In [ ]:
REPO = 'https://github.com/Shashank17singh/TruthLens.git'
BRANCH = 'main'
import os

# Kaggle's "Save & Run All" runs headless for up to 12 hours with no idle timeout; Colab free stops after about 90 idle minutes.
KAGGLE = os.path.exists('/kaggle/working')
HOME = '/tmp/TruthLens' if KAGGLE else '/content/TruthLens'  # /tmp on Kaggle: /kaggle/working is capped at 20 GB
# Each training command stops cleanly after the epoch that would overrun this many minutes; rerunning it resumes.
BUDGET = 600 if KAGGLE else 75
if not os.path.exists(HOME):
    !git clone -q --branch {BRANCH} {REPO} {HOME}
%cd {HOME}
!git pull -q
assert os.path.exists('ml/audio/train.py'), 'This branch does not have ml/audio/train.py yet. Push your commits, then run this cell again.'

In [ ]:
!pip install -q -r ml/requirements.txt
import torch

GPU = torch.cuda.is_available()
print(torch.cuda.get_device_name(0) if GPU else 'No GPU: cells that need one will stop. Runtime, Change runtime type, T4 GPU.')

If pip asks you to restart the session, restart it and run the two cells above again. The clone is kept.

In [ ]:
if KAGGLE:
    DRIVE = '/kaggle/working/truthlens'  # saved as the notebook's output; download it from the Output tab
else:
    from google.colab import drive

    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/truthlens'
os.makedirs(DRIVE, exist_ok=True)
try:
    if KAGGLE:
        from kaggle_secrets import UserSecretsClient

        os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
    else:
        from google.colab import userdata

        os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
except Exception:
    print('No KAGGLE_API_TOKEN secret: the download falls back to the slower official server.')

## Data

ASVspoof 2019 LA is 7.6 GB. It comes from the Kaggle copy when a `KAGGLE_API_TOKEN` Colab secret is set, which is much faster, and otherwise from the official server, which drops long transfers, so that download resumes until the file is complete. Every member's CRC is checked before use.

The prepared audio and manifest are saved to Drive as one tar, so after a disconnect these two cells restore it instead of downloading and preparing again. Delete the tar on Drive to start over.

In [ ]:
import glob
import shutil
import tarfile

RAW = 'data/raw/asvspoof2019_la'
CACHE = f'{DRIVE}/asvspoof2019_la_prepared.tar'
PREPARED = ['data/processed/asvspoof2019_la', 'data/manifests/asvspoof2019_la.csv',
            'data/manifests/asvspoof2019_la.balance.json', 'docs/datacards/asvspoof2019_la.md',
            f'{RAW}/LA/ASVspoof2019_LA_asv_scores']  # evaluation reads the official ASV scores

if os.path.exists(CACHE):
    print('Restoring the prepared audio from Drive')
    with tarfile.open(CACHE) as bundle:
        bundle.extractall('.', filter='data')
else:
    os.makedirs(RAW, exist_ok=True)
    if os.environ.get('KAGGLE_API_TOKEN'):
        !kaggle datasets download -d hahunavth/asvspoof2019-la -p {RAW} -q
    else:
        SIZE = 7640952520
        URL = 'https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip'
        for attempt in range(200):
            if os.path.exists(f'{RAW}/LA.zip') and os.path.getsize(f'{RAW}/LA.zip') >= SIZE:
                break
            !curl -sS -L -C - --max-time 1800 -o {RAW}/LA.zip {URL}
    ARCHIVE = glob.glob(f'{RAW}/*.zip')[0]
    print(ARCHIVE, os.path.getsize(ARCHIVE), 'bytes')

In [ ]:
import zipfile

if not os.path.exists(CACHE):
    with zipfile.ZipFile(ARCHIVE) as bundle:
        bad = bundle.testzip()
        assert any(n.startswith('LA/ASVspoof2019_LA_cm_protocols/') for n in bundle.namelist()), 'Unexpected layout: no LA/ folder'
    assert bad is None, f'Corrupt member {bad}. Delete the zip and download again.'
    !python -m ml.common.prepare asvspoof2019_la --workers 4
    # Write under a temporary name so a disconnect mid-copy never leaves a half tar that looks complete.
    with tarfile.open(CACHE + '.partial', 'w') as bundle:
        for path in PREPARED:
            if os.path.exists(path):
                bundle.add(path)
    os.replace(CACHE + '.partial', CACHE)
    os.remove(ARCHIVE)  # keep the extracted ASV scores, which evaluation needs
print(sum(1 for _ in open('data/manifests/asvspoof2019_la.csv')) - 1, 'clips in the manifest')

## Train

LCNN with LFCC is the model the service deploys, and it also trains on a CPU runtime. wav2vec2 and WavLM need the GPU. Each command resumes after a disconnect.

In [ ]:
CKPT = f'{DRIVE}/checkpoints/audio'
!python -m ml.audio.train model=lcnn feature=lfcc checkpoint_dir={CKPT} tracker=none max_minutes={BUDGET} num_workers=2

In [ ]:
if not GPU:
    print('Skipped: this cell needs a GPU runtime.')
else:
    !python -m ml.audio.train model=wav2vec2 checkpoint_dir={CKPT} tracker=none max_minutes={BUDGET} num_workers=2 steps_per_epoch=2000

In [ ]:
if not GPU:
    print('Skipped: this cell needs a GPU runtime.')
else:
    !python -m ml.audio.train model=wavlm checkpoint_dir={CKPT} tracker=none max_minutes={BUDGET} num_workers=2 steps_per_epoch=2000

## Evaluate and export

With the official ASV scores, the report adds min t-DCF for the evaluation set.

In [ ]:
import glob

ASV = f'{RAW}/LA/ASVspoof2019_LA_asv_scores/ASVspoof2019.LA.asv.eval.gi.trl.scores.txt'
for best in sorted(glob.glob(f'{CKPT}/*/*-best.pt')):
    run = os.path.basename(os.path.dirname(best))
    !python -m ml.audio.evaluate {best} --asv-scores {ASV} --output {DRIVE}/results/audio/{run}

In [ ]:
LCNN = glob.glob(f'{CKPT}/audio-lcnn-lfcc-*/*-best.pt')[0]
!python -m ml.audio.export {LCNN} --output-dir {DRIVE}/onnx

## Hand back the results

This packs every run's metrics and figures, the ONNX export, the manifest balance and the data card into one zip on Drive. Download it and send it over to wire the model into the app.

In [ ]:
import shutil

PACK = f'{DRIVE}/truthlens_audio_results'
shutil.rmtree(PACK, ignore_errors=True)
shutil.copytree(f'{DRIVE}/results/audio', f'{PACK}/results/audio')
shutil.copytree(f'{DRIVE}/onnx', f'{PACK}/onnx')
shutil.copy('data/manifests/asvspoof2019_la.balance.json', f'{PACK}/')
if os.path.exists('docs/datacards/asvspoof2019_la.md'):
    shutil.copy('docs/datacards/asvspoof2019_la.md', f'{PACK}/')
shutil.make_archive(PACK, 'zip', PACK)
print('Download', PACK + '.zip')